# Lab 9 — Declarative Pipeline Development with Delta Live Tables / Lakeflow Declarative Pipelines

**Goal:** Define Bronze ingestion, Silver quality expectations, and a Gold aggregate.

**Important platform requirement:** This notebook contains pipeline source code intended to be configured as a Databricks Lakeflow Declarative Pipelines (formerly Delta Live Tables) pipeline. It is **not** expected to run as a normal notebook attached to general-purpose compute. Configure the pipeline in the Databricks UI and set `SOURCE_PATH` to a supported cloud-storage or Unity Catalog Volume path. The `dlt` library is supplied by the pipeline runtime.

## Pipeline source code

Create a separate pipeline source notebook in Databricks and paste/adapt the following code there. Configure the pipeline's target catalog/schema and source path in the pipeline settings. Keep the source path outside `/tmp` for production-style cloud ingestion.

In [0]:
!pip install dlt

In [0]:

from pyspark.sql.functions import (
    col, to_date, sum as spark_sum, expr
)

BRONZE_TABLE = "workspace.lab_db.bronze_orders_lab6"
SILVER_TABLE = "workspace.lab_db.silver_orders_lab9"
GOLD_TABLE = "workspace.lab_db.gold_daily_sales_lab9"

# Bronze -> Silver: read, cast, validate and deduplicate
bronze = spark.table(BRONZE_TABLE)

silver = (
    bronze
    .withColumn("amount", expr("try_cast(amount AS DOUBLE)"))
    .withColumn("customer_id", expr("try_cast(customer_id AS INT)"))
    .withColumn("order_timestamp", col("order_timestamp").cast("timestamp"))
    .filter(
        col("amount").isNotNull()
        & (col("amount") > 0)
        & col("customer_id").isNotNull()
        & col("order_id").isNotNull()
        & col("order_timestamp").isNotNull()
    )
    .dropDuplicates(["order_id"])
)

(silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE))

# Silver -> Gold: aggregate daily revenue per customer
silver = spark.table(SILVER_TABLE)

gold = (
    silver
    .withColumn("order_date", to_date(col("order_timestamp")))
    .groupBy("order_date", "customer_id")
    .agg(spark_sum("amount").alias("total_spent"))
)

(gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLE))

display(spark.table(GOLD_TABLE).orderBy("order_date", "customer_id"))


## Pipeline setup checklist

1. In the Databricks workspace, open **Jobs & Pipelines** and create a pipeline.
2. Add the source notebook containing the code above.
3. Set the target catalog/schema and choose an available pipeline mode.
4. Replace the placeholder `SOURCE_PATH` with your actual Volume/cloud-storage path.
5. Start the pipeline and inspect the event log, expectations, and output tables.

If your workspace does not expose Lakeflow Declarative Pipelines, document the limitation and complete the concept review using the Bronze/Silver/Gold notebooks (Labs 6–8).